In [1]:
# ============================================================
# CUSTOMER CHURN - MULTIPLE ML MODELS
# TRAIN, COMPARE AND SAVE BEST MODEL
# ============================================================

# Install if required:
# pip install pandas numpy scikit-learn xgboost joblib

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)


# ============================================================
# 1. LOAD DATASET
# ============================================================

df = pd.read_csv("/content/customer_churn_ml_dataset.csv")

print("\nDataset loaded successfully!")
print("Shape:", df.shape)

print("\nFirst 5 rows:")
print(df.head())


# ============================================================
# 2. CHECK DATA
# ============================================================

print("\nMissing values:")
print(df.isnull().sum())

print("\nChurn distribution:")
print(df["Churn"].value_counts())


# ============================================================
# 3. REMOVE CUSTOMER ID
# ============================================================

df = df.drop("CustomerID", axis=1)


# ============================================================
# 4. DEFINE FEATURES AND TARGET
# ============================================================

X = df.drop("Churn", axis=1)

# Convert Yes/No into 1/0
y = df["Churn"].map({
    "No": 0,
    "Yes": 1
})


# ============================================================
# 5. IDENTIFY NUMERICAL AND CATEGORICAL COLUMNS
# ============================================================

numeric_features = [
    "Age",
    "Tenure_Months",
    "Monthly_Charges",
    "Total_Charges",
    "Support_Calls"
]

categorical_features = [
    "Contract",
    "Internet_Service",
    "Payment_Method"
]


# ============================================================
# 6. PREPROCESSING
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)


# ============================================================
# 7. TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


# ============================================================
# 8. DEFINE ML MODELS
# ============================================================

models = {

    "Logistic Regression":
        LogisticRegression(max_iter=1000),

    "Decision Tree":
        DecisionTreeClassifier(
            random_state=42
        ),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=200,
            random_state=42
        ),

    "Gradient Boosting":
        GradientBoostingClassifier(
            random_state=42
        ),

    "KNN":
        KNeighborsClassifier(
            n_neighbors=5
        ),

    "SVM":
        SVC(
            kernel="rbf",
            probability=True,
            random_state=42
        )
}


# ============================================================
# 9. TRAIN AND EVALUATE MODELS
# ============================================================

results = []

best_model = None
best_model_name = None
best_accuracy = 0


for name, model in models.items():

    print("\n" + "=" * 60)
    print("Training:", name)
    print("=" * 60)

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    # Train
    pipeline.fit(X_train, y_train)

    # Predict
    y_pred = pipeline.predict(X_test)

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    print("Accuracy :", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1 Score :", round(f1, 4))

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })


    # Check whether this is the best model
    if accuracy > best_accuracy:

        best_accuracy = accuracy
        best_model = pipeline
        best_model_name = name


# ============================================================
# 10. MODEL COMPARISON
# ============================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
)

print("\n\n")
print("=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

print(
    results_df.to_string(
        index=False
    )
)


# ============================================================
# 11. BEST MODEL
# ============================================================

print("\n")
print("=" * 70)
print("BEST MODEL")
print("=" * 70)

print("Model:", best_model_name)
print("Accuracy:", round(best_accuracy * 100, 2), "%")


# ============================================================
# 12. SAVE BEST MODEL
# ============================================================

model_filename = "best_churn_model.pkl"

joblib.dump(
    best_model,
    model_filename
)

print("\nBest model saved successfully!")
print("File:", model_filename)


# ============================================================
# 13. SAVE MODEL COMPARISON
# ============================================================

results_df.to_csv(
    "model_comparison.csv",
    index=False
)

print("Model comparison saved as: model_comparison.csv")


# ============================================================
# 14. FINAL CLASSIFICATION REPORT
# ============================================================

best_predictions = best_model.predict(X_test)

print("\n")
print("=" * 70)
print("CLASSIFICATION REPORT - BEST MODEL")
print("=" * 70)

print(
    classification_report(
        y_test,
        best_predictions,
        target_names=["No Churn", "Churn"]
    )
)


Dataset loaded successfully!
Shape: (1000, 10)

First 5 rows:
  CustomerID  Age  Tenure_Months  Monthly_Charges  Total_Charges  \
0   CUST0001   56              2           114.28         254.18   
1   CUST0002   69              2            83.27         177.73   
2   CUST0003   46              8           133.55         946.78   
3   CUST0004   32              6           137.09         921.85   
4   CUST0005   60              4            74.82         260.35   

   Support_Calls        Contract Internet_Service    Payment_Method Churn  
0              1  Month-to-month              DSL       Credit card   Yes  
1              0        One year               No       Credit card    No  
2              4  Month-to-month               No       Credit card   Yes  
3              2  Month-to-month      Fiber optic  Electronic check   Yes  
4              3  Month-to-month      Fiber optic       Credit card    No  

Missing values:
CustomerID          0
Age                 0
Tenure_Mont